## XML Generation with Glom Detection:

In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
import openslide
from openslide import OpenSlide, deepzoom
import xml.etree.ElementTree as ET
from PIL import Image
import tifffile as tiff
import sys
import logging
from datetime import datetime
from tqdm import tqdm
import gc
from scipy.ndimage import gaussian_filter
from skimage.morphology import area_opening, area_closing
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev

# Configure GPU memory growth to avoid OOM errors
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    for device in physical_devices:
        try:
            tf.config.experimental.set_memory_growth(device, True)
        except Exception as e:
            print(f"Error setting memory growth: {e}")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("wsi_processing.log"),
        logging.StreamHandler()
    ]
)

def clear_memory():
    """Clear memory between operations to avoid OOM errors"""
    gc.collect()
    tf.keras.backend.clear_session()
    for _ in range(3):
        gc.collect()

def DAB_detect(tile, sensitivity=1.2):
    """
    Detect DAB (brown) staining in a tile with enhanced sensitivity
    
    Args:
        tile: RGB image tile from the WSI
        sensitivity: Multiplier to increase detection sensitivity
        
    Returns:
        mask_DAB: Binary mask of detected DAB staining
    """
    # Use more inclusive HSV range for better detection of faint staining
    light_br = (0, 0, 0)
    dark_br = (120, 255, 180)  # Expanded range for better detection
    
    test_hsv = cv2.cvtColor(tile, cv2.COLOR_RGB2HSV)
    mask_br = cv2.inRange(test_hsv, light_br, dark_br)
    
    # Use smaller sigma for less smoothing to preserve small details
    mask_gaussian_PTC = gaussian_filter(mask_br, 0.8)
    
    # Lower threshold to detect more subtle staining
    threshold_value = int(100 / sensitivity)
    ret, thresh = cv2.threshold(mask_gaussian_PTC, threshold_value, 255, cv2.THRESH_BINARY)

    # Reduce area thresholds to preserve smaller structures
    tile_open = area_opening(thresh, area_threshold=40, connectivity=1)  # Reduced from 80
    mask_DAB = area_closing(tile_open, area_threshold=200, connectivity=1)  # Reduced from 400
    
    return mask_DAB

def is_background(tile_img, threshold=8):
    """
    Always returns False to process all tiles regardless of background
    """
    return False  # Remove background check

def load_model(model_path, custom_objects=None):
    """
    Load a TensorFlow model with error handling
    
    Args:
        model_path: Path to the model file
        custom_objects: Dictionary of custom objects for model loading
        
    Returns:
        model: Loaded TensorFlow model
    """
    logging.info(f"Loading model from: {model_path}")
    try:
        model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
        logging.info("Model loaded successfully")
        return model
    except Exception as e:
        logging.error(f"Error loading model: {e}")
        raise

def generate_masks_and_save_tiff(wsi_path, model, mask_tiff_path, brown_stains_tiff_path, patch_size=512, mag_level=2, batch_size=16):
    """
    Process a whole slide image, generate masks using a model, and save as TIFF
    
    Args:
        wsi_path: Path to the whole slide image
        model: TensorFlow model for prediction
        mask_tiff_path: Output path for the mask TIFF
        brown_stains_tiff_path: Output path for the DAB stain TIFF
        patch_size: Size of patches to process
        mag_level: Magnification level to use
        batch_size: Batch size for model prediction
    """
    # Load the WSI
    logging.info(f"Loading WSI: {wsi_path}")
    Slide = OpenSlide(wsi_path)
    
    # Get the dimensions of the WSI at the specified magnification level
    img_size = Slide.level_dimensions[mag_level]
    logging.info(f"Image size at zoom level {mag_level}: {img_size}")
    
    # Create empty placeholder images for the full mask and DAB-detected mask
    mask_placeholder = Image.new('1', img_size)  # Binary mask (mode '1' for 1-bit pixels)
    mask_placeholder_1 = Image.new('1', img_size)  # For DAB detection
    
    # Scaling factor (from mag_level to full resolution)
    factor = 2 ** mag_level
    
    # Calculate overlay for better stitching and to avoid edge artifacts
    overlay = patch_size // 8  # Creating 12.5% overlap between tiles
    
    # Loop through the image by tiles with overlap
    cols = (img_size[0] // (patch_size - overlay)) + 1  # Add 1 to ensure coverage
    rows = (img_size[1] // (patch_size - overlay)) + 1
    
    total_tiles = cols * rows
    logging.info(f"Processing {total_tiles} tiles with {overlay}px overlap")
    
    # Prepare batches for more efficient processing
    batch_tiles = []
    
    with tqdm(total=total_tiles, desc="Processing tiles") as pbar:
        for row in range(rows):
            for col in range(cols):
                # Calculate top-left corner of the tile with overlap
                x = col * (patch_size - overlay)
                y = row * (patch_size - overlay)
                
                # Ensure we don't exceed image boundaries
                w = min(patch_size, img_size[0] - x)
                h = min(patch_size, img_size[1] - y)
                
                if w <= 0 or h <= 0:
                    continue  # Skip if outside image boundaries

                # Read the tile from the WSI at the specified magnification level
                tile = Slide.read_region((x * factor, y * factor), mag_level, (w, h))
                tile_np = np.array(tile.convert("RGB"))  # Convert to NumPy array

                # For tiles smaller than patch_size, pad them to full size
                if w < patch_size or h < patch_size:
                    pad_tile = np.zeros((patch_size, patch_size, 3), dtype=np.uint8)
                    pad_tile[:h, :w, :] = tile_np
                    tile_np = pad_tile

                # Add to batch for processing (no background check)
                batch_tiles.append((x, y, w, h, tile_np))
                
                # Process batch when it reaches the batch size or at the end
                if len(batch_tiles) == batch_size or (row == rows-1 and col == cols-1):
                    process_tile_batch(batch_tiles, model, mask_placeholder, mask_placeholder_1, patch_size)
                    pbar.update(len(batch_tiles))
                    batch_tiles = []
                    
                    # Periodically clear memory
                    if len(batch_tiles) % 1000 == 0:
                        clear_memory()
    
    logging.info(f"Saving masks to TIFF files: {mask_tiff_path}, {brown_stains_tiff_path}")
    
    # Save the stitched masks as TIFF files
    mask_placeholder.save(mask_tiff_path)
    mask_placeholder_1.save(brown_stains_tiff_path)
    logging.info("TIFF files saved successfully")

def process_tile_batch(batch_tiles, model, mask_placeholder, mask_placeholder_1, patch_size):
    """
    Process a batch of tiles with the model
    
    Args:
        batch_tiles: List of tuples (x, y, w, h, tile_np) to process
        model: TensorFlow model for prediction
        mask_placeholder: PIL Image for storing prediction masks
        mask_placeholder_1: PIL Image for storing DAB stain masks
        patch_size: Size of the tiles
    """
    if not batch_tiles:
        return
        
    # Prepare batch for model
    batch_resized = []
    for _, _, _, _, tile_np in batch_tiles:
        # Resize the tile to 224x224 before sending it to the model
        tile_resized = cv2.resize(tile_np, (224, 224), interpolation=cv2.INTER_NEAREST)
        tile_resized = tile_resized / 255.0  # Normalize to [0, 1]
        batch_resized.append(tile_resized)
    
    batch_processed = np.array(batch_resized, dtype=np.float32)
    
    # Predict the masks using the model
    # Always use 0.4 as threshold
    predicted_masks = model.predict(batch_processed) > 0.4
    
    # Process predictions and update the mask placeholders
    for i, (x, y, w, h, tile_np) in enumerate(batch_tiles):
        # Get the prediction for this tile
        predicted_mask = predicted_masks[i]
        predicted_mask = (predicted_mask.squeeze() * 255).astype(np.uint8)  # Convert to 0-255 mask
        
        # Resize the predicted mask back to patch_size to fit the WSI tile size
        predicted_mask_resized = cv2.resize(predicted_mask, (patch_size, patch_size), interpolation=cv2.INTER_NEAREST)
        
        # Use only the valid part if this was a partial tile
        valid_mask = predicted_mask_resized[:h, :w]
        
        # Convert the predicted mask to a PIL image (binary format)
        mask_image = Image.fromarray(valid_mask).convert('1')

        # Paste the mask onto the full-sized mask placeholder
        mask_placeholder.paste(mask_image, (x, y))
        
        # Detect DAB in the tile with increased sensitivity
        mask_1 = DAB_detect(tile_np, sensitivity=1.2)
        
        # Use only the valid part of the DAB mask for partial tiles
        if mask_1.shape[:2] != (h, w):
            mask_1 = mask_1[:h, :w]
            
        mask_1 = mask_1.astype(np.uint8)
        mask_image_1 = Image.fromarray(mask_1).convert('1')
        
        # Paste the DAB mask onto the second placeholder
        mask_placeholder_1.paste(mask_image_1, (x, y))

def xml_create():
    """
    Create a new XML tree for annotations
    
    Returns:
        Annotations: XML Element for annotations
    """
    Annotations = ET.Element('Annotations')
    return Annotations

def xml_add_annotation(Annotations, fixed_color=65280, annotationID=None):
    """
    Add annotation to XML
    
    Args:
        Annotations: XML Element for annotations
        fixed_color: Color for the annotation
        annotationID: ID for the annotation
        
    Returns:
        Annotations: Updated XML Element
    """
    if annotationID is None:  # Not specified
        annotationID = len(Annotations.findall('Annotation')) + 1
    
    Annotation = ET.SubElement(Annotations, 'Annotation', attrib={
        'Type': '4',
        'Visible': '1',
        'ReadOnly': '0',
        'Incremental': '0',
        'LineColorReadOnly': '0',
        'LineColor': str(fixed_color),  # Use the fixed color for all annotations
        'Id': str(annotationID),
        'NameReadOnly': '0'
    })
    
    Attributes = ET.SubElement(Annotation, 'Attributes')
    ET.SubElement(Attributes, 'Attribute', attrib={
        'Name': 'Description',
        'Id': '0',
        'Value': ''
    })
    Regions = ET.SubElement(Annotation, 'Regions')
    return Annotations

def xml_add_region(Annotations, pointList, annotationID=-1, regionID=None, scale_x=1, scale_y=1):
    """
    Add region to annotation
    
    Args:
        Annotations: XML Element for annotations
        pointList: List of points for the region
        annotationID: ID for the annotation
        regionID: ID for the region
        scale_x: X scaling factor
        scale_y: Y scaling factor
        
    Returns:
        Annotations: Updated XML Element
    """
    Annotation = Annotations.find(f"Annotation[@Id='{annotationID}']")
    Regions = Annotation.find('Regions')
    if regionID is None:  # Not specified
        regionID = len(Regions.findall('Region')) + 1
    Region = ET.SubElement(Regions, 'Region', attrib={
        'NegativeROA': '0',
        'ImageFocus': '-1',
        'DisplayId': '1',
        'InputRegionId': '0',
        'Analyze': '0',
        'Type': '0',
        'Id': str(regionID)
    })
    Vertices = ET.SubElement(Region, 'Vertices')
    for point in pointList:
        ET.SubElement(Vertices, 'Vertex', attrib={
            'X': str(point['X']*scale_x),
            'Y': str(point['Y']*scale_y),
            'Z': '0'
        })
    # Ensure contour is closed by repeating the first point
    ET.SubElement(Vertices, 'Vertex', attrib={
        'X': str(pointList[0]['X']*scale_x),
        'Y': str(pointList[0]['Y']*scale_y),
        'Z': '0'
    })
    return Annotations

def load_existing_polygons(xml_file_path):
    """
    Load existing polygons from an XML file
    
    Args:
        xml_file_path: Path to the XML file
        
    Returns:
        list: List of polygons
    """
    try:
        # Parse the XML file and extract polygon coordinates
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        
        polygons = []
        for annotation in root.findall('.//Annotation'):
            for region in annotation.findall('.//Region'):
                vertices = region.find('Vertices')
                polygon = []
                for vertex in vertices.findall('Vertex'):
                    x = float(vertex.get('X'))
                    y = float(vertex.get('Y'))
                    polygon.append((x, y))
                polygons.append(np.array(polygon, dtype=np.float32))
        
        logging.info(f"Loaded {len(polygons)} polygons from {xml_file_path}")
        return polygons
    except Exception as e:
        logging.warning(f"Error loading polygons from {xml_file_path}: {e}")
        return []

def remove_gloms(contour, existing_polygons):
    """
    Check if a contour overlaps with existing polygons
    
    Args:
        contour: Contour to check
        existing_polygons: List of existing polygons
        
    Returns:
        boolean: True if overlap exists, False otherwise
    """
    if not existing_polygons:
        return False
        
    contour_poly = cv2.convexHull(contour).reshape(-1, 2)
    for existing_polygon in existing_polygons:
        # Check each point in the contour to see if it's inside any existing polygon
        for point in contour_poly:
            point_tuple = (float(point[0]), float(point[1]))
            if cv2.pointPolygonTest(existing_polygon, point_tuple, False) >= 0:
                return True
    return False

def indent_xml(elem, level=0):
    """
    In-place indentation of an ElementTree XML element.
    This function is designed for compatibility with Python versions before 3.9,
    which don't have the built-in ET.indent() function.
    
    Args:
        elem: XML element to indent
        level: Current indentation level
    """
    i = "\n" + level * "  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
        for child in elem:
            indent_xml(child, level + 1)
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
    else:
        if level and (not elem.tail or not elem.tail.strip()):
            elem.tail = i

def convert_tiff_to_xml(mask_tiff_path, output_xml_path, base_name, wsi_path):
    """
    Convert a TIFF mask to XML annotations
    
    Args:
        mask_tiff_path: Path to the mask TIFF
        output_xml_path: Output path for the XML
        base_name: Base name for the file
        wsi_path: Path to the whole slide image
    """
    Image.MAX_IMAGE_PIXELS = None
    
    logging.info(f"Converting TIFF to XML: {mask_tiff_path} -> {output_xml_path}")
    
    try:
        # Load the WSI to get full resolution dimensions
        Slide = OpenSlide(wsi_path)
        original_width, original_height = Slide.level_dimensions[0]  # Full resolution (level 0)
        zoom_level_width, zoom_level_height = Slide.level_dimensions[2]  # Current zoom level dimensions

        # Calculate scaling factors based on magnification level
        scale_x = original_width / zoom_level_width
        scale_y = original_height / zoom_level_height
        
        # Load the TIFF file
        mask_image = Image.open(mask_tiff_path)
        
        # Try to load existing glomeruli polygons for exclusion
        glom_xml_path = f"/blue/pinaki.sarder/surajramadugu/MVI/PTC_Classify/glom_xml/{base_name}.xml"
        existing_polygons = []
        if os.path.exists(glom_xml_path):
            existing_polygons = load_existing_polygons(glom_xml_path)
        
        # Convert image to 'L' mode (8-bit grayscale) if it's in '1' mode
        if mask_image.mode == '1':
            mask_image = mask_image.convert('L')
        
        # Convert the PIL image to a Numpy array
        mask_np = np.array(mask_image)

        # Ensure the Numpy array is in uint8 format
        if mask_np.dtype != np.uint8:
            mask_np = mask_np.astype(np.uint8)
        
        # Apply Gaussian blur (no reduction)
        blurred_mask = cv2.GaussianBlur(mask_np, ksize=(3,3), sigmaX=1.5)

        # Threshold the blurred mask to convert it back to binary
        _, binary_mask = cv2.threshold(blurred_mask, 127, 255, cv2.THRESH_BINARY)
        
        # Find contours in the binary mask
        # Use RETR_TREE to capture hierarchical contours
        contours, hierarchy = cv2.findContours(binary_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_TC89_KCOS)
        logging.info(f"Found {len(contours)} contours in the mask")

        # Initialize XML root
        Annotations = xml_create()
        
        # Process each contour and create the XML structure
        annotationID = 1
        Annotations = xml_add_annotation(Annotations=Annotations, fixed_color=65280, annotationID=annotationID)

        regionID = 1  # Initialize the region ID
        accepted_contours = 0
        rejected_gloms = 0
        
        for contour in contours:
            # Accept all contours, no minimum point check
            
            # Scale the contour points to match the original image dimensions
            s_contour = (contour.reshape(-1, 2) * np.array([scale_x, scale_y])).astype(np.float32)
            scaled_contour = s_contour.astype(np.int32).reshape(-1, 1, 2)
            
            # Check if the contour lies within any existing polygon (e.g., glomeruli)
            # KEEPING this check as requested
            if remove_gloms(scaled_contour, existing_polygons):
                rejected_gloms += 1
                continue  # Skip this contour

            # No contour simplification - keep all points
            # Convert contour to a point list format expected by xml_add_region
            pointList = [{'X': int(point[0][0]), 'Y': int(point[0][1])} for point in contour]

            # Add the region (contour) to the annotation
            Annotations = xml_add_region(Annotations=Annotations, pointList=pointList, 
                                        annotationID=annotationID, regionID=regionID, 
                                        scale_x=scale_x, scale_y=scale_y)

            # Increment the region ID for the next contour
            regionID += 1
            accepted_contours += 1

        logging.info(f"Accepted {accepted_contours} contours")
        logging.info(f"Rejected {rejected_gloms} contours that overlapped with glomeruli")

        # Write the XML to a file
        tree = ET.ElementTree(Annotations)
        
        # Format the XML with indentation for readability
        try:
            # Use built-in indent function if available (Python 3.9+)
            ET.indent(tree, space="  ")
        except AttributeError:
            # Fall back to our custom indent function for older Python versions
            indent_xml(Annotations)
        
        tree.write(output_xml_path, encoding="UTF-8", xml_declaration=True)
        logging.info(f"XML file generated and saved to {output_xml_path}")
        
    except Exception as e:
        logging.error(f"Error in convert_tiff_to_xml: {e}")
        raise

def process_wsi(wsi_path, model_path, output_dir, custom_objects=None):
    """
    Process a whole slide image, generate masks, and save as TIFF and XML
    
    Args:
        wsi_path: Path to the whole slide image
        model_path: Path to the model
        output_dir: Output directory
        custom_objects: Custom objects for model loading
    """
    start_time = datetime.now()
    logging.info(f"Starting processing at {start_time}")
    
    try:
        # Create output directories if they don't exist
        mask_tiff_dir = os.path.join(output_dir, "masks_PTC")
        brown_stains_tiff_dir = os.path.join(output_dir, "masks_brown_stain")
        output_xml_dir = os.path.join(output_dir, "xml_generated")
        
        for directory in [mask_tiff_dir, brown_stains_tiff_dir, output_xml_dir]:
            os.makedirs(directory, exist_ok=True)
        
        # Extract the base name of the image file
        base_name = os.path.splitext(os.path.basename(wsi_path))[0]
        
        # Define output paths with clean absolute paths
        mask_tiff_path = os.path.join(mask_tiff_dir, f"{base_name}.tiff")
        brown_stains_tiff_path = os.path.join(brown_stains_tiff_dir, f"{base_name}.tiff")
        output_xml_path = os.path.join(output_xml_dir, f"{base_name}.xml")
        
        # Load the model
        model = load_model(model_path, custom_objects)
        
        # Generate masks and save as TIFF
        generate_masks_and_save_tiff(
            wsi_path=wsi_path,
            model=model,
            mask_tiff_path=mask_tiff_path,
            brown_stains_tiff_path=brown_stains_tiff_path,
        )
        
        # Convert the mask to XML
        convert_tiff_to_xml(
            mask_tiff_path=mask_tiff_path,
            output_xml_path=output_xml_path,
            base_name=base_name,
            wsi_path=wsi_path
        )
        
        elapsed = datetime.now() - start_time
        logging.info(f"Processing completed in {elapsed}")
        
    except Exception as e:
        logging.error(f"Error processing WSI: {e}")
        raise
    finally:
        clear_memory()

def main():
    """Main function to process multiple WSI images"""
    # Import custom metrics if needed
    try:
        from metrics import dice_loss, dice_coef, iou
        custom_objects = {'dice_loss': dice_loss, 'dice_coef': dice_coef, 'iou': iou}
    except ImportError:
        logging.warning("Could not import custom metrics, will try to load model without them")
        custom_objects = None
    
    # Define images to process
    image_filenames = [
        "S07-43516CD34LCA.ndpi"
    ]
    
    # Common directory path for all images
    common_directory = "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images"
    
    # Load the model
    model_path = "best_model.h5"
    
    # Define output directory with leading slash
    output_dir = "/home/durgasrit.dongla/UNet_PTC/PTC_IPTCR/generated_xml"
    
    # Process each image
    for image_filename in image_filenames:
        try:
            wsi_path = os.path.join(common_directory, image_filename)
            logging.info(f"Processing WSI: {image_filename}")
            
            process_wsi(wsi_path, model_path, output_dir, custom_objects)
            
            logging.info(f"Completed processing: {image_filename}")
        except Exception as e:
            logging.error(f"Error processing {image_filename}: {e}")
            continue

if __name__ == "__main__":
    main()

## Create the XML for the brown stains along with finding IPTCR values

In [8]:
import os
import numpy as np
import cv2
from openslide import OpenSlide
import xml.etree.ElementTree as ET
from PIL import Image
import logging
from tqdm import tqdm
import concurrent.futures
import matplotlib.pyplot as plt
from IPython.display import display

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def load_ptc_xml(xml_path):
    """
    Load PTC regions from XML annotation file
    
    Args:
        xml_path: Path to the XML annotation file
        
    Returns:
        list: List of contours (each contour is a numpy array of points)
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        contours = []
        for annotation in root.findall('.//Annotation'):
            for region in annotation.findall('.//Region'):
                vertices = region.find('Vertices')
                if vertices is None:
                    continue
                    
                points = []
                for vertex in vertices.findall('Vertex'):
                    x = float(vertex.get('X'))
                    y = float(vertex.get('Y'))
                    points.append([x, y])
                
                if len(points) >= 3:  # Need at least 3 points to form a contour
                    contours.append(np.array(points, dtype=np.int32))
        
        print(f"Loaded {len(contours)} PTC contours from {xml_path}")
        return contours
    except Exception as e:
        print(f"Error loading XML file {xml_path}: {e}")
        return []

def process_patch(args):
    """
    Process a single patch for brown stain detection
    
    Args:
        args: Tuple containing (slide, x, y, level, patch_size, factor)
        
    Returns:
        tuple: (x, y, detected_mask)
    """
    slide, x, y, level, patch_size, cur_width, cur_height, factor = args
    
    # Read patch from slide
    patch = slide.read_region((x * factor, y * factor), level, (cur_width, cur_height))
    patch_np = np.array(patch.convert("RGB"))
    
    # Detect brown stains
    detected_mask = detect_brown_in_patch(patch_np)
    
    return (x, y, detected_mask)

def detect_brown_stains(wsi_path, level=2, patch_size=1024, overlap=0, threads=8):
    """
    Detect brown stains in a whole slide image using parallel processing
    
    Args:
        wsi_path: Path to the whole slide image
        level: Magnification level to use
        patch_size: Size of patches to process
        overlap: Overlap between patches
        threads: Number of parallel threads
        
    Returns:
        list: List of detected brown stain contours
        np.ndarray: Binary mask of brown stains
    """
    # Load the WSI
    slide = OpenSlide(wsi_path)
    
    # Get dimensions
    width, height = slide.level_dimensions[level]
    
    # Create empty mask for brown stains
    mask = np.zeros((height, width), dtype=np.uint8)
    
    # Calculate step size with overlap
    step_size = patch_size - overlap
    
    # Calculate scaling factor from level 0 to current level
    factor = 2 ** level
    
    # Prepare arguments for parallel processing
    args_list = []
    for y in range(0, height, step_size):
        for x in range(0, width, step_size):
            # Adjust patch size if it exceeds boundaries
            cur_width = min(patch_size, width - x)
            cur_height = min(patch_size, height - y)
            
            if cur_width <= 0 or cur_height <= 0:
                continue
            
            args_list.append((slide, x, y, level, patch_size, cur_width, cur_height, factor))
    
    # Process patches in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=threads) as executor:
        results = list(tqdm(executor.map(process_patch, args_list), 
                           total=len(args_list), 
                           desc="Detecting brown stains"))
    
    # Combine the results
    for x, y, detected_mask in results:
        h, w = detected_mask.shape
        mask[y:y+h, x:x+w] = detected_mask
    
    # Find contours in the mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_KCOS)
    
    # Filter small contours (likely noise)
    min_area = 100
    max_area = 15000  # This threshold filters large DAB aggregates
    filtered_contours = [c for c in contours if min_area <= cv2.contourArea(c) <= max_area]

    
    print(f"Detected {len(filtered_contours)} brown stain contours")
    
    # Scale contours to match level 0 coordinates
    scaled_contours = []
    for contour in filtered_contours:
        scaled_contour = contour.copy()
        scaled_contour[:, :, 0] = scaled_contour[:, :, 0] * factor
        scaled_contour[:, :, 1] = scaled_contour[:, :, 1] * factor
        scaled_contours.append(scaled_contour)
    
    return scaled_contours, mask

def detect_brown_in_patch(patch):
    """
    Detect brown staining in a patch using HSV color thresholding
    
    Args:
        patch: RGB image patch
        
    Returns:
        np.ndarray: Binary mask of detected brown stains
    """
    # Convert to HSV color space
    hsv = cv2.cvtColor(patch, cv2.COLOR_RGB2HSV)
    
    # Define brown color range in HSV
    lower_brown = np.array([0, 50, 20])
    upper_brown = np.array([30, 255, 200])
    
    # Create mask for brown pixels
    brown_mask = cv2.inRange(hsv, lower_brown, upper_brown)
    
    # Apply morphological operations to clean up
    kernel = np.ones((3, 3), np.uint8)
    cleaned = cv2.morphologyEx(brown_mask, cv2.MORPH_OPEN, kernel)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
    
    return cleaned

def save_brown_stain_xml(contours, output_path, color=16711680):  # 16711680 is blue in BGR format
    """
    Save brown stain contours as XML annotations
    
    Args:
        contours: List of contours
        output_path: Path to save the XML file
        color: Color for the annotations (in BGR format)
    """
    # Create XML structure
    root = ET.Element("Annotations")
    
    # Add annotation
    annotation = ET.SubElement(root, "Annotation", {
        "Type": "4",
        "Visible": "1",
        "ReadOnly": "0",
        "Incremental": "0",
        "LineColorReadOnly": "0",
        "LineColor": str(color),  # Blue color (BGR format)
        "Id": "1",
        "NameReadOnly": "0"
    })
    
    # Add attributes
    attributes = ET.SubElement(annotation, "Attributes")
    ET.SubElement(attributes, "Attribute", {
        "Name": "Description",
        "Id": "0",
        "Value": "Brown Stains"
    })
    
    # Add regions
    regions = ET.SubElement(annotation, "Regions")
    
    # Add each contour as a region
    for i, contour in enumerate(contours):
        region = ET.SubElement(regions, "Region", {
            "NegativeROA": "0",
            "ImageFocus": "-1",
            "DisplayId": "1",
            "InputRegionId": "0",
            "Analyze": "0",
            "Type": "0",
            "Id": str(i + 1)
        })
        
        # Add vertices
        vertices = ET.SubElement(region, "Vertices")
        
        # Add each point as a vertex
        for point in contour:
            x, y = point[0]
            ET.SubElement(vertices, "Vertex", {
                "X": str(float(x)),
                "Y": str(float(y)),
                "Z": "0"
            })
        
        # Close the polygon by adding the first point again
        if len(contour) > 0:
            x, y = contour[0][0]
            ET.SubElement(vertices, "Vertex", {
                "X": str(float(x)),
                "Y": str(float(y)),
                "Z": "0"
            })
    
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Write to file
    tree = ET.ElementTree(root)
    tree.write(output_path, encoding="UTF-8", xml_declaration=True)
    print(f"Saved brown stain annotations to {output_path}")

def check_ptc_overlap(args):
    """
    Check if a PTC overlaps with brown stain mask
    
    Args:
        args: Tuple containing (ptc, factor, brown_mask, width, height, min_overlap_percent)
        
    Returns:
        tuple: (ptc_index, overlap_found, overlap_percentage)
    """
    ptc, ptc_index, factor, brown_mask, width, height, min_overlap_percent = args
    
    # Scale PTC contour to detection level
    scaled_ptc = ptc.copy().astype(np.float32)
    scaled_ptc[:, 0] = scaled_ptc[:, 0] * factor
    scaled_ptc[:, 1] = scaled_ptc[:, 1] * factor
    scaled_ptc = scaled_ptc.astype(np.int32)
    
    # Create a smaller mask just for this PTC
    x, y, w, h = cv2.boundingRect(scaled_ptc)
    
    # Check bounds
    if x >= width or y >= height:
        return (ptc_index, False, 0)
        
    # Adjust width and height to stay within bounds
    w = min(w, width - x)
    h = min(h, height - y)
    
    if w <= 0 or h <= 0:
        return (ptc_index, False, 0)
    
    # Create mask for this PTC
    ptc_mask = np.zeros((h, w), dtype=np.uint8)
    
    # Adjust contour coordinates to the cropped region
    adjusted_ptc = scaled_ptc.copy()
    adjusted_ptc[:, 0] = adjusted_ptc[:, 0] - x
    adjusted_ptc[:, 1] = adjusted_ptc[:, 1] - y
    
    # Fill the contour
    cv2.fillPoly(ptc_mask, [adjusted_ptc], 255)
    
    # Get the corresponding region from the brown mask
    try:
        brown_region = brown_mask[y:y+h, x:x+w]
    except IndexError:
        # Handle case where the region is out of bounds
        return (ptc_index, False, 0)
    
    # Check overlap
    overlap = cv2.bitwise_and(ptc_mask, brown_region)
    total_pixels = np.count_nonzero(ptc_mask)
    overlap_pixels = np.count_nonzero(overlap)
    
    overlap_percentage = (overlap_pixels / total_pixels * 100) if total_pixels > 0 else 0
    
    # Only count as overlap if it exceeds minimum percentage
    overlap_found = overlap_percentage >= min_overlap_percent
    
    return (ptc_index, overlap_found, overlap_percentage)

def calculate_overlap_optimized(ptc_contours, brown_mask, slide_dims, level, threads=8, min_overlap_percent=15):
    """
    Calculate overlap between PTC contours and brown stain mask using parallel processing
    
    Args:
        ptc_contours: List of PTC contours at level 0
        brown_mask: Binary mask of brown stains at detection level
        slide_dims: Dimensions of the slide at detection level
        level: Detection level
        threads: Number of parallel threads
        min_overlap_percent: Minimum percentage of PTC area that must overlap to count
        
    Returns:
        float: Percentage of PTCs that overlap with brown stains
        list: List of overlapping PTC indices
        list: List of overlap percentages for each PTC
    """
    width, height = slide_dims
    
    # Factor to scale from level 0 to detection level
    factor = 1 / (2 ** level)
    
    # Prepare arguments for parallel processing
    args_list = []
    for i, ptc in enumerate(ptc_contours):
        args_list.append((ptc, i, factor, brown_mask, width, height, min_overlap_percent))
    
    # Process PTCs in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=threads) as executor:
        results = list(tqdm(executor.map(check_ptc_overlap, args_list), 
                           total=len(args_list), 
                           desc="Checking PTC overlaps"))
    
    # Parse results
    overlapping_ptcs = []
    overlap_percentages = [0] * len(ptc_contours)
    
    for ptc_index, overlap_found, overlap_percentage in results:
        overlap_percentages[ptc_index] = overlap_percentage
        if overlap_found:
            overlapping_ptcs.append(ptc_index)
    
    # Calculate overall percentage
    overall_percentage = (len(overlapping_ptcs) / len(ptc_contours)) * 100 if len(ptc_contours) > 0 else 0
    
    return overall_percentage, overlapping_ptcs, overlap_percentages

def main(wsi_path, ptc_xml_path, output_dir, detection_level=4, threads=8, min_overlap_percent=15, display_count=5):
 
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get base name for output
    base_name = os.path.splitext(os.path.basename(wsi_path))[0]
    
    print(f"Processing {base_name}")
    
    # Load PTC contours
    ptc_contours = load_ptc_xml(ptc_xml_path)
    
    if len(ptc_contours) == 0:
        print("No PTCs found in XML file. Exiting.")
        return {
            "total_ptcs": 0,
            "overlapping_ptcs": 0,
            "overlap_percentage": 0,
            "error": "No PTCs found in XML file"
        }
    
    # Detect brown stains
    print("Detecting brown stains...")
    brown_contours, brown_mask = detect_brown_stains(
        wsi_path, level=detection_level, threads=threads
    )
    
    # Save brown stain annotations as XML
    brown_xml_path = os.path.join(output_dir, f"{base_name}_brown_stains.xml")
    save_brown_stain_xml(brown_contours, brown_xml_path)
    
    # Calculate overlap
    print("Calculating overlap...")
    slide = OpenSlide(wsi_path)
    width, height = slide.level_dimensions[detection_level]
    
    overlap_percentage, overlap_indices, overlap_percentages = calculate_overlap_optimized(
        ptc_contours, brown_mask, (width, height), detection_level, 
        threads=threads, min_overlap_percent=min_overlap_percent
    )
    
    print(f"Overlap analysis complete:")
    print(f"Total PTCs: {len(ptc_contours)}")
    print(f"PTCs with significant brown stain overlap (≥{min_overlap_percent}%): {len(overlap_indices)}")
    print(f"Overlap percentage: {overlap_percentage:.8f}%".format(overlap_percentage))

# Example usage
if __name__ == "__main__":
    wsi_path = "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images/S15-69213CD34LCA.ndpi"
    ptc_xml_path = "/orange/pinaki.sarder/durgasrit.dongla/PTC_IPTCR/Predicted_XML/xml_generated/S15-69213CD34LCA.xml"
    output_dir = "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/PTC_IPTCR"
    
    # Run with higher detection level for faster processing
    results = main(
        wsi_path, 
        ptc_xml_path, 
        output_dir,
        detection_level=2,     
        threads=8,             # Use 8 threads for parallel processing
        min_overlap_percent=3
    )

Processing S15-69213CD34LCA
Loaded 1966 PTC contours from /orange/pinaki.sarder/durgasrit.dongla/PTC_IPTCR/Predicted_XML/xml_generated/S15-69213CD34LCA.xml
Detecting brown stains...


Detecting brown stains: 100%|██████████| 88/88 [00:00<00:00, 216.85it/s]


Detected 1007 brown stain contours
Saved brown stain annotations to /blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/PTC_IPTCR/S15-69213CD34LCA_brown_stains.xml
Calculating overlap...


Checking PTC overlaps: 100%|██████████| 1966/1966 [00:00<00:00, 14550.42it/s]

Overlap analysis complete:
Total PTCs: 1966
PTCs with significant brown stain overlap (≥3%): 694
Overlap percentage: 35.30010173%


In [10]:
import os
import numpy as np
import cv2
import xml.etree.ElementTree as ET
from openslide import OpenSlide
from tqdm import tqdm


def load_ptc_xml(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        contours = []
        for annotation in root.findall('.//Annotation'):
            for region in annotation.findall('.//Region'):
                vertices = region.find('Vertices')
                if vertices is None:
                    continue
                points = []
                for vertex in vertices.findall('Vertex'):
                    x = float(vertex.get('X'))
                    y = float(vertex.get('Y'))
                    points.append([x, y])
                if len(points) >= 3:
                    contours.append(np.array(points, dtype=np.int32))
        return contours
    except Exception as e:
        print(f"Error loading XML file {xml_path}: {e}")
        return []


def detect_brown_stains(wsi_path, level=2, patch_size=1024):
    slide = OpenSlide(wsi_path)
    width, height = slide.level_dimensions[level]
    mask = np.zeros((height, width), dtype=np.uint8)
    step_size = patch_size
    factor = 2 ** level

    for y in tqdm(range(0, height, step_size), desc="DAB Detection Rows"):
        for x in range(0, width, step_size):
            patch = slide.read_region((x * factor, y * factor), level, (patch_size, patch_size))
            patch_np = np.array(patch.convert("RGB"))
            hsv = cv2.cvtColor(patch_np, cv2.COLOR_RGB2HSV)
            lower_brown = np.array([0, 0, 0])
            upper_brown = np.array([100, 255, 160])
            brown_mask = cv2.inRange(hsv, lower_brown, upper_brown)
            kernel = np.ones((3, 3), np.uint8)
            cleaned = cv2.morphologyEx(brown_mask, cv2.MORPH_OPEN, kernel)
            cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)

            h, w = cleaned.shape
            max_h = min(h, mask.shape[0] - y)
            max_w = min(w, mask.shape[1] - x)
            mask[y:y+max_h, x:x+max_w] = cleaned[:max_h, :max_w]

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_area = 10
    max_area = 1500000
    filtered_contours = [c for c in contours if min_area <= cv2.contourArea(c) <= max_area]
    scaled_contours = []
    for contour in filtered_contours:
        scaled = contour.astype(np.float32)
        scaled[:, :, 0] *= factor
        scaled[:, :, 1] *= factor
        scaled_contours.append(scaled.astype(np.int32))
    return scaled_contours, mask


def calculate_iptcr(ptc_contours, brown_mask, slide_dims, level=2, min_overlap_percent=3):
    factor = 1 / (2 ** level)
    width, height = slide_dims

    overlap_indices = []
    for idx, ptc in enumerate(ptc_contours):
        scaled_ptc = ptc.astype(np.float32)
        scaled_ptc[:, 0] *= factor
        scaled_ptc[:, 1] *= factor
        scaled_ptc = scaled_ptc.astype(np.int32)
        x, y, w, h = cv2.boundingRect(scaled_ptc)
        if x >= width or y >= height:
            continue
        w = min(w, width - x)
        h = min(h, height - y)
        ptc_mask = np.zeros((h, w), dtype=np.uint8)
        adjusted = scaled_ptc.copy()
        adjusted[:, 0] -= x
        adjusted[:, 1] -= y
        cv2.fillPoly(ptc_mask, [adjusted], 255)
        brown_region = brown_mask[y:y+h, x:x+w]
        overlap = cv2.bitwise_and(ptc_mask, brown_region)
        overlap_ratio = np.count_nonzero(overlap) / np.count_nonzero(ptc_mask) * 100 if np.count_nonzero(ptc_mask) > 0 else 0
        if overlap_ratio >= min_overlap_percent:
            overlap_indices.append(idx)

    iptcr_score = len(overlap_indices) / len(ptc_contours) if ptc_contours else 0
    return iptcr_score, len(overlap_indices), len(ptc_contours)


def export_xml(filename, layers):
    root = ET.Element("Annotations")
    for i, (name, contours, color) in enumerate(layers):
        annotation = ET.SubElement(root, "Annotation", {
            "Id": str(i+1),
            "Name": name,
            "LineColor": str(color),
            "Visible": "1",
            "ReadOnly": "0",
            "Type": "4"
        })
        ET.SubElement(annotation, "Attributes")
        regions = ET.SubElement(annotation, "Regions")
        for ridx, contour in enumerate(contours):
            region = ET.SubElement(regions, "Region", {
                "Id": str(ridx+1), "Type": "0", "NegativeROA": "0", "InputRegionId": "0", "Analyze": "0"
            })
            vertices = ET.SubElement(region, "Vertices")
            for pt in contour:
                x, y = pt[0] if len(pt.shape) == 2 else pt
                ET.SubElement(vertices, "Vertex", {
                    "X": str(x),
                    "Y": str(y),
                    "Z": "0"
                })
            first_pt = contour[0][0] if len(contour[0].shape) == 2 else contour[0]
            ET.SubElement(vertices, "Vertex", {
                "X": str(first_pt[0]),
                "Y": str(first_pt[1]),
                "Z": "0"
            })
    tree = ET.ElementTree(root)
    tree.write(filename, encoding="utf-8", xml_declaration=True)


def main(wsi_path, ptc_xml_path):
    slide = OpenSlide(wsi_path)
    ptc_contours = load_ptc_xml(ptc_xml_path)
    if not ptc_contours:
        print("No PTC contours found.")
        return

    print("Detecting brown-stained regions...")
    brown_contours, brown_mask = detect_brown_stains(wsi_path)

    print("Filtering out PTCs inside large brown areas...")
    large_dab_contours = [c for c in brown_contours if cv2.contourArea(c) > 15000]
    dab_mask = np.zeros(slide.level_dimensions[0][::-1], dtype=np.uint8)
    for contour in large_dab_contours:
        cv2.fillPoly(dab_mask, [contour], 255)

    ptc_selected, ptc_rejected = [], []
    for ptc in ptc_contours:
        M = cv2.moments(ptc)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        if 0 <= cy < dab_mask.shape[0] and 0 <= cx < dab_mask.shape[1] and dab_mask[cy, cx] > 0:
            ptc_rejected.append(ptc)
        else:
            ptc_selected.append(ptc)

    dab_selected = [c for c in brown_contours if cv2.contourArea(c) <= 15000]
    dab_rejected = [c for c in brown_contours if cv2.contourArea(c) > 15000]

    print(f"Excluded {len(ptc_rejected)} PTCs due to full DAB overlap.")

    print("Calculating IPTCR score...")
    iptcr_score, n_overlap, n_total = calculate_iptcr(
        ptc_selected, brown_mask, slide.level_dimensions[2], level=2)

    print(f"Total PTCs: {len(ptc_contours)}")
    print(f"Remaining after exclusion: {len(ptc_selected)}")
    print(f"Overlapping PTCs: {n_overlap}")
    print(f"IPTCR Score: {iptcr_score:.4f}")

    print("Exporting XML with four layers...")
    base_name = os.path.splitext(os.path.basename(wsi_path))[0]
    output_xml_path = base_name + "_layers.xml"
    export_xml(output_xml_path, [
        ("PTC Selected", ptc_selected, 65280),
        ("DAB Selected", dab_selected, 255),
        ("DAB Rejected", dab_rejected, 16711680),
        ("PTC Rejected", ptc_rejected, 16776960)
    ])
    print(f"XML saved to {output_xml_path}")


if __name__ == "__main__":
    wsi_path = "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images/S15-69213CD34LCA.ndpi"
    ptc_xml_path = "/orange/pinaki.sarder/durgasrit.dongla/PTC_IPTCR/Predicted_XML/xml_generated/S15-69213CD34LCA.xml"
    main(wsi_path, ptc_xml_path)


Detecting brown-stained regions...


DAB Detection Rows: 100%|██████████| 11/11 [00:04<00:00,  2.44it/s]


Filtering out PTCs inside large brown areas...
Excluded 32 PTCs due to full DAB overlap.
Calculating IPTCR score...
Total PTCs: 1966
Remaining after exclusion: 1934
Overlapping PTCs: 571
IPTCR Score: 0.2952
Exporting XML with four layers...
XML saved to S15-69213CD34LCA_layers.xml


In [5]:
import os

if __name__ == "__main__":
    # Define directories
    xml_dir = "/orange/pinaki.sarder/durgasrit.dongla/PTC_IPTCR/Predicted_XML/xml_generated"
    images_dir = "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images"
    output_dir =  "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/PTC_IPTCR"
    
    # Get all XML files in the directory
    xml_files = [f for f in os.listdir(xml_dir) if f.endswith('.xml')]
    
    # Process each XML file
    for xml_file in xml_files:
        # Extract base filename (without extension)
        base_name = os.path.splitext(xml_file)[0]
        
        # Construct the corresponding image path
        possible_image_paths = [
            os.path.join(images_dir, f"{base_name}.ndpi"),  # .ndpi extension
        ]
        
        # Find which image file exists
        wsi_path = None
        for possible_path in possible_image_paths:
            if os.path.exists(possible_path):
                wsi_path = possible_path
                break
        
        if wsi_path is None:
            continue
        
        ptc_xml_path = os.path.join(xml_dir, xml_file)
        
        print(f"Processing {wsi_path} with {xml_file}...")
        
        # Run with higher detection level for faster processing
        results = main(
            wsi_path, 
            ptc_xml_path, 
            output_dir,
            detection_level=2,     
            threads=8,             # Use 8 threads for parallel processing
            min_overlap_percent=3
        )
        
        print(f"Completed processing {wsi_path}")

Processing /blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images/S15-56749CD34LCA.ndpi with S15-56749CD34LCA.xml...
Processing S15-56749CD34LCA
Loaded 4317 PTC contours from /orange/pinaki.sarder/durgasrit.dongla/PTC_IPTCR/Predicted_XML/xml_generated/S15-56749CD34LCA.xml
Detecting brown stains...


Detecting brown stains: 100%|██████████| 77/77 [00:05<00:00, 15.31it/s]


Detected 456 brown stain contours
Saved brown stain annotations to /blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/PTC_IPTCR/S15-56749CD34LCA_brown_stains.xml
Calculating overlap...


Checking PTC overlaps: 100%|██████████| 4317/4317 [00:00<00:00, 16099.21it/s]


Overlap analysis complete:
Total PTCs: 4317
PTCs with significant brown stain overlap (≥3%): 1019
Overlap percentage: 23.60435488%
Completed processing /blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images/S15-56749CD34LCA.ndpi
Processing /blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images/S15-69213CD34LCA.ndpi with S15-69213CD34LCA.xml...
Processing S15-69213CD34LCA
Loaded 1966 PTC contours from /orange/pinaki.sarder/durgasrit.dongla/PTC_IPTCR/Predicted_XML/xml_generated/S15-69213CD34LCA.xml
Detecting brown stains...


Detecting brown stains: 100%|██████████| 88/88 [00:00<00:00, 182.91it/s]


... (121 outputs left out)

Detecting brown stains: 100%|██████████| 105/105 [00:00<00:00, 212.82it/s]


Detected 121 brown stain contours
Saved brown stain annotations to /blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/PTC_IPTCR/S07-43516CD34LCA_brown_stains.xml
Calculating overlap...


Checking PTC overlaps: 100%|██████████| 1545/1545 [00:00<00:00, 16819.11it/s]

Overlap analysis complete:
Total PTCs: 1545
PTCs with significant brown stain overlap (≥3%): 296
Overlap percentage: 19.15857605%
Completed processing /blue/pinaki.sarder/akshitagupta/MicrovascularInf/Final_Data/Images/S07-43516CD34LCA.ndpi
